# 1. Preprocessing Overview

**Project:** Predicting Rural Water Point Functionality Using Data Mining

**Target variable:** `functional3` (Functional / Partially functional / Abandoned or not functional)

**Task:** Multiclass classification

**Raw dataset:** `data/raw/water_pump_functionality.xlsx` (read-only, never modified)

This notebook implements **Step 5 - Data Preprocessing, Feature Engineering and Feature Selection**. It turns the decisions documented and approved in [`reports/preprocessing_decisions.md`](../reports/preprocessing_decisions.md) into working code, informed by the findings of [`notebooks/01_data_understanding.ipynb`](01_data_understanding.ipynb) and [`notebooks/02_eda.ipynb`](02_eda.ipynb).

**Guiding principle - no test-set leakage.** Every statistic used anywhere in this notebook (medians, category-frequency thresholds, bin edges, encoders, scalers) is fitted on the **training split only**, then applied unchanged to the test split. The fixed workflow (matching decision-plan Section 9) is:

```
Raw data
    |
    v
Remove target leakage / identifier / administrative columns (fixed rule, no fitting)
    |
    v
Separate X and y
    |
    v
Train / Test split (stratified on functional3, random_state=42, test_size=0.20)
    |
    +--> Training split ----------------+--> Testing split
         |                              |
         v                              v
   Fit ALL data-dependent steps    Apply the SAME fitted
   on the training split ONLY      transformations only
         |                              |
         v                              v
   Transform training data         Transform testing data
```

**This notebook does NOT:** train a model, tune hyperparameters, apply SMOTE/resampling, compare algorithms, or build the backend/frontend/API. It stops once the processed datasets and the fitted preprocessing pipeline are saved and validated.

# 2. Load Raw Dataset

Load the raw Excel file exactly as in the previous two notebooks. The file is treated as **read-only**: every subsequent step operates on in-memory copies (`df.copy()`, or new `DataFrame`s produced by `.drop()` / transformers), and the original file on disk is never written to.

In [1]:
import os
import numpy as np
import pandas as pd
import joblib
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, OrdinalEncoder, StandardScaler

DATA_PATH = "../data/raw/water_pump_functionality.xlsx"
TARGET = "functional3"
CLASS_ORDER = ["Functional", "Partially functional", "Abandoned or not functional"]
RANDOM_STATE = 42
TEST_SIZE = 0.20

# Snapshot the raw file's modification time and size; re-checked in Section 16 to
# confirm the raw Excel file on disk was never modified by this notebook.
_raw_stat_before = os.stat(DATA_PATH)

df_raw = pd.read_excel(DATA_PATH)
print(f"Raw dataset loaded: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns")
df_raw.head()

Raw dataset loaded: 1793 rows, 52 columns


,id,country,admin1,admin2,admin3,instance_wp,subdate_wp,latitude_wp,longitude_wp,elevation_wp,...,qtyhh_c_wp,wc_admin_index_wp,wc_finance_index_wp,wc_mgmt_index_wp,wc_maint_index_wp,wc_savings_wp,wpqty_wp,pop_1000,annual_rain,season
0,1,Ethiopia,Tigray,North Western,Medebay Zana,762331022,2020-01-02,14.116315,38.493830,2033.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,Ethiopia,Tigray,North Western,Tselemti,751461172,2019-12-21,13.581540,38.467207,1428.1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,Ethiopia,Tigray,North Western,Medebay Zana,758401118,2019-12-29,14.097803,38.452013,1997.3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,Ethiopia,Tigray,North Western,Medebay Zana,769951024,2019-12-29,13.912499,38.437931,1877.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,Ethiopia,Tigray,North Western,Tahtay Koraro,765940926,2019-12-18,14.041615,38.437492,1966.3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [2]:
# Work on a copy from this point on -- df_raw itself is never modified or re-saved.
df = df_raw.copy()
assert df_raw.shape == (1793, 52), "Unexpected raw shape -- verify the source file"
print("Copy created for preprocessing. df_raw remains untouched (read-only reference).")

Copy created for preprocessing. df_raw remains untouched (read-only reference).


# 3. Define Target and Features

`functional3` is the target (`y`). Every other raw column is a *candidate* predictor at this stage; leakage, identifier and administrative columns are removed explicitly in Section 4 before `X` is finalized.

In [3]:
assert TARGET in df.columns, f"Target column '{TARGET}' not found"

y_full = df[TARGET].copy()
X_full_candidates = df.drop(columns=[TARGET]).copy()

print(f"Target: '{TARGET}'")
print(f"Candidate predictor columns (before leakage/identifier removal): {X_full_candidates.shape[1]}")
print()
print("Target class counts (full raw dataset):")
print(y_full.value_counts().reindex(CLASS_ORDER))

Target: 'functional3'
Candidate predictor columns (before leakage/identifier removal): 51

Target class counts (full raw dataset):
functional3
Functional                     1333
Partially functional            157
Abandoned or not functional     303
Name: count, dtype: int64


# 4. Remove Leakage and Identifier Variables

This is a **fixed, rule-based** removal -- it uses no statistic computed from the data, so it is safe to apply before the train/test split (it cannot leak information between splits; it only removes columns that should never be model inputs at all).

### A. Target leakage variables (decision-plan Section 2)

These describe **current/recent water availability** -- i.e. they are symptoms or restatements of `functional3` itself, not independent predictors:

| Variable | Reason for exclusion |
|---|---|
| `wateravailable` | Near-perfect split of the target (EDA Sec. 12): "No" -> 300 Abandoned + 2 Partially + 0 Functional; "Yes" -> 1,333 Functional + 155 Partially + 3 Abandoned. |
| `whynowatertoday_wp` | Only ever filled in when there is currently no water -- its *missingness itself* almost perfectly reveals the target class (non-missing for 300/303 Abandoned, 2/157 Partially, 0/1,333 Functional). |
| `whynowatertoday_wp_other` | Free-text follow-up to the field above; same leakage mechanism, and 99.0% missing besides. |
| `downtime2weeks` | "Yes" -> 272 Abandoned + 81 Partially + 1 Functional; "No" -> 30 Abandoned + 76 Partially + 1,332 Functional -- a direct downtime symptom of the target. |
| `minutesfill20l` | Mean 1.45 min for Functional vs. 98.4 min for Partially functional; almost no data at all for Abandoned. A direct symptom of a struggling/non-functional source. |
| `brokendays_wp` | Mean 429.8 days (Abandoned) vs. 27.2 (Functional) vs. 50.0 (Partially functional) -- a quantified restatement of non-functionality. |
| `qtymonthsnowater_wp` | Mean 2.51 months (Abandoned) vs. 0.13 (Functional) vs. 0.53 (Partially functional) -- same reasoning. |

`lockedfullday_wp` was investigated in the EDA for the same reason and was **not** found to behave like a leakage variable (18.4% vs. 11.7% abandoned across its two levels -- no sharp separation), so it is kept as a candidate predictor.

### B. Identifier / survey-design / high-cardinality administrative variables (decision-plan Sections 2, 7-D)

| Variable | Reason for exclusion |
|---|---|
| `id` | Row-level identifier, 1,793/1,793 unique. Carries no generalizable signal. |
| `instance_wp` | Survey-instance identifier, 1,793/1,793 unique. |
| `photo_wp` | Photo URL/identifier, 1,792/1,793 unique. |
| `commid_wp` | Community identifier, 1,395/1,793 unique -- too fine-grained to generalize. |
| `cwid` | Committee identifier, 948 unique, 46.5% missing. |
| `timepoint_wp` | Encodes which *survey wave/sample* a record belongs to (a study-design artifact -- mean `wp_age` is 1.77 yrs for "new construction" vs. 5.87 yrs for "older construction" records), not a property of the water point itself. |
| `dataorg_wp` | Maps 1-to-1 onto `country` (every `ET.RST` record is Ethiopia, etc.) -- functionally redundant; `country` is kept instead. |
| `subdate_wp` | Survey submission date (338 unique date strings). Categorized under decision-plan Section 8, Category D (identifier / survey-design columns to exclude). It is not itself a property of the water point and no feature-engineering use for it was approved in the decision plan, so it is excluded here even though it is not individually re-listed in the Step 5 task brief -- flagged again in Section 17 (Unresolved Issues) for visibility. |
| `admin2`, `admin3` | High-cardinality administrative subdivisions (30 and 108 categories respectively, the latter 11.2% missing) relative to a 1,793-row dataset -- one-hot encoding `admin3` alone would add >100 columns. `country` and `admin1` (15 categories, 0% missing) are kept as the geographic predictors, per the task brief. |

### C. Columns flagged for further investigation, not yet cleared for inclusion (decision-plan Section 8, Category E)

`grant_number_wp` (69.5% missing; only 10 distinct values -- looks like a program code, not a continuous variable) and `balance_any_dollars_wp` (47.6% missing) are listed in the decision plan as **"requires further investigation before inclusion"**, separate from the approved predictor candidates in Category B. The Step 5 task brief does not instruct cleaning, engineering, or including either column. Per "do not silently introduce unrelated preprocessing decisions," they are **excluded here pending that investigation**, rather than silently included or silently dropped without a paper trail -- also flagged in Section 17.

All other candidate predictors are carried forward into the train/test split.

In [4]:
LEAKAGE_COLS = [
    "wateravailable",
    "whynowatertoday_wp",
    "whynowatertoday_wp_other",
    "downtime2weeks",
    "minutesfill20l",
    "brokendays_wp",
    "qtymonthsnowater_wp",
]

IDENTIFIER_ADMIN_COLS = [
    "id",
    "instance_wp",
    "photo_wp",
    "commid_wp",
    "cwid",
    "timepoint_wp",
    "dataorg_wp",
    "subdate_wp",
    "admin2",
    "admin3",
]

PENDING_INVESTIGATION_COLS = [
    "grant_number_wp",
    "balance_any_dollars_wp",
]

DROP_COLS = LEAKAGE_COLS + IDENTIFIER_ADMIN_COLS + PENDING_INVESTIGATION_COLS
assert len(DROP_COLS) == len(set(DROP_COLS)), "Duplicate column in drop list"

missing_from_df = [c for c in DROP_COLS if c not in df.columns]
assert not missing_from_df, f"Expected columns not found in raw data: {missing_from_df}"

y = df[TARGET].copy()
X = df.drop(columns=[TARGET] + DROP_COLS).copy()

print(f"Leakage columns removed: {len(LEAKAGE_COLS)}")
print(f"Identifier/administrative columns removed: {len(IDENTIFIER_ADMIN_COLS)}")
print(f"Pending-investigation columns removed: {len(PENDING_INVESTIGATION_COLS)}")
print(f"Total columns removed (excluding target): {len(DROP_COLS)}")
print(f"X shape after removal: {X.shape}")

assert TARGET not in X.columns
assert all(c not in X.columns for c in DROP_COLS)
print("\nVerified: target and all excluded columns are absent from X.")

Leakage columns removed: 7
Identifier/administrative columns removed: 10
Pending-investigation columns removed: 2
Total columns removed (excluding target): 19
X shape after removal: (1793, 32)

Verified: target and all excluded columns are absent from X.


# 5. Train / Test Split

Split **before** any data-dependent preprocessing is fitted (decision-plan Section 9). This is a stratified split on `functional3`, using `random_state=42` and `test_size=0.20`, so that:

- No imputation statistic, category-frequency threshold, encoder, scaler, or engineered-feature statistic computed later ever sees a test-split row.
- The class imbalance identified in the EDA (74.3% / 8.8% / 16.9%) is preserved proportionally in both splits.

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"y_train: {y_train.shape}, y_test: {y_test.shape}")
assert X_train.shape[0] + X_test.shape[0] == X.shape[0]
assert X_train.shape[0] == y_train.shape[0]
assert X_test.shape[0] == y_test.shape[0]

X_train: (1434, 32), X_test: (359, 32)
y_train: (1434,), y_test: (359,)


In [6]:
# Class distribution: full dataset vs. train vs. test
dist_full = (y.value_counts(normalize=True) * 100).reindex(CLASS_ORDER).round(2)
dist_train = (y_train.value_counts(normalize=True) * 100).reindex(CLASS_ORDER).round(2)
dist_test = (y_test.value_counts(normalize=True) * 100).reindex(CLASS_ORDER).round(2)

count_full = y.value_counts().reindex(CLASS_ORDER)
count_train = y_train.value_counts().reindex(CLASS_ORDER)
count_test = y_test.value_counts().reindex(CLASS_ORDER)

class_dist_report = pd.DataFrame({
    "full_count": count_full, "full_pct": dist_full,
    "train_count": count_train, "train_pct": dist_train,
    "test_count": count_test, "test_pct": dist_test,
})
class_dist_report

,full_count,full_pct,train_count,train_pct,test_count,test_pct
functional3,,,,,,
Functional,1333,74.34,1066,74.34,267,74.37
Partially functional,157,8.76,126,8.79,31,8.64
Abandoned or not functional,303,16.90,242,16.88,61,16.99


**Observation:** the stratified split keeps class proportions closely aligned across the full dataset, training split, and test split (each class within roughly +/-0.2 percentage points), confirming the stratification worked as intended.

# 6. Data Cleaning

## Invalid-value cleanup: `wc_savings_wp`

The decision plan (Section 1 row 3, Section 3) identified that `wc_savings_wp` mixes genuine binary values (1.0 = has savings, 0.0 = no savings) with two **survey sentinel codes** that are not real savings magnitudes:

- `999.0` (n=84 in the full dataset) -- "don't know" / not a real value
- `888.0` (n=2 in the full dataset) -- "other" / not a real value

These are recoded to missing (`NaN`) **before** any imputation statistic (e.g. a median) is computed for this column -- otherwise a handful of "999" values would badly distort any mean/median/scaling computed on it.

This is a **deterministic, fixed-rule replacement** (999 -> NaN, 888 -> NaN) -- it uses no statistic derived from the data, so, unlike imputation, it is safe to apply identically and independently to `X_train` and `X_test` after the split (no information crosses between the splits).

In [7]:
SENTINEL_MAP = {999.0: np.nan, 888.0: np.nan}


def clean_wc_savings(X):
    # Recode wc_savings_wp survey sentinel codes (999, 888) to missing.
    X = X.copy()
    X["wc_savings_wp"] = X["wc_savings_wp"].replace(SENTINEL_MAP)
    return X


print("wc_savings_wp value counts BEFORE cleanup (train):")
print(X_train["wc_savings_wp"].value_counts(dropna=False).sort_index())

X_train = clean_wc_savings(X_train)
X_test = clean_wc_savings(X_test)

print("\nwc_savings_wp value counts AFTER cleanup (train):")
print(X_train["wc_savings_wp"].value_counts(dropna=False).sort_index())

assert not (X_train["wc_savings_wp"] == 999.0).any()
assert not (X_train["wc_savings_wp"] == 888.0).any()
assert not (X_test["wc_savings_wp"] == 999.0).any()
assert not (X_test["wc_savings_wp"] == 888.0).any()
print("\nVerified: no 999/888 sentinel values remain in wc_savings_wp (train or test).")

wc_savings_wp value counts BEFORE cleanup (train):
wc_savings_wp
0.0      192
1.0      678
888.0      1
999.0     65
NaN      498
Name: count, dtype: int64

wc_savings_wp value counts AFTER cleanup (train):
wc_savings_wp
0.0    192
1.0    678
NaN    564
Name: count, dtype: int64

Verified: no 999/888 sentinel values remain in wc_savings_wp (train or test).


## Duplicate rows

The EDA notebook (Section 6) confirmed 0 fully duplicated rows in the raw dataset (0.0%). No duplicate-removal action is needed.

## Outliers

Per the decision plan (Section 4), **no outlier removal or capping is performed**. Extreme values are treated as follows:

- `wp_age` (up to 74 years), `pop_1000` (up to ~9,413), `brokendays_wp` (already excluded as leakage): kept as potentially valid extreme observations. `wp_age`'s skew is addressed later via a log1p transform for scale-sensitive algorithms (Section 10), not by removing points.
- `qtypeople_wp` / `qtyhh_wp` (up to 5,000 / 800) and `pumpstrokes` (up to 95): flagged as "requires further investigation" in the EDA, but no verified correction was approved, so raw values are kept unmodified.
- `elevation_wp` (min -217 m; 107 negative values, 106 of them in West Bengal, India): **not corrected**. The concentration of negative values in one region is a plausible data-quality/DEM-lookup artifact, but no verified reference-elevation correction has been approved in the decision plan. The variable is kept as-is; **the suspicious negative values require further investigation against an external elevation source before any correction is made** (flagged again in Section 17).

# 7. Missing-Value Handling

Per decision-plan Section 3, **a single blanket strategy is not used**. The 32 remaining predictor columns are grouped by *why* they are missing, and each group gets a different, documented treatment. All statistics referenced below (medians, "most frequent" thresholds) are fitted **inside the pipeline in Section 12, on the training split only** -- this section documents the strategy and performs the two steps that do not depend on any fitted statistic (they are fixed, deterministic rules, so they are applied directly here).

| Missingness pattern | Columns | Mechanism | Strategy |
|---|---|---|---|
| **Structurally missing (categorical, conditional)** | `pumptype`, `drillmethod`, `piped_source`, `piped_pump` | Only applies to certain `wptype` values (e.g. a protected spring has no pump type) -- EDA Section 5/14. | Explicit **"Not applicable"** category, not mode imputation (imputing a real pump type would misrepresent water points where no pump exists). |
| **Structurally missing (numeric, conditional)** | `rehab_age` | Missing for 100% of records where `rehabyn` = "No" (EDA Section 5) -- the value does not apply, it is not unknown. | Fill with `0` **plus** an explicit `rehab_age_missing` indicator column (Section 8), so the model can distinguish "never rehabilitated" from "rehabilitated 0 years ago". |
| **Moderate/high missingness, plausibly informative (categorical)** | `wc_present_wp`, `paytocollect_wp`, `season`, `wc_admin_index_wp`, `wc_finance_index_wp`, `wc_mgmt_index_wp`, `wc_maint_index_wp` | Missingness appears batch-like (several columns share close to the same missing count), not obviously conditional on another field. | Explicit **"Unknown"** category (or, for the ordinal `wc_*_index_wp` variables, a dedicated missing code, Section 9), never silently imputed into an existing real category. |
| **Moderate/high missingness, plausibly informative (numeric)** | `annual_rain`, `wpqty_wp`, `qtyhh_c_wp`, `improved_wponly_wp`, `wc_savings_wp`, `pumpstrokes` | Same batch-like pattern; `pumpstrokes` (45.6% missing) is not individually itemized in decision-plan Section 3, so it is treated with the same conservative "moderate/high missingness" numeric rule rather than being silently equated with the low-missingness columns below. | **Median imputation (train-fit) plus a missing-indicator column**, so "imputed" stays distinguishable from "observed". |
| **Low missingness, likely closer to random (categorical)** | `whomanage_wp` (0.06%), `rehabyn` (20.47% -- not itemized in decision-plan Section 3's table, treated conservatively as its own explicit category rather than assumed random), `cwfunded_wp` (0.73%), `lockedfullday_wp` (4.02%), `country`, `admin1`, `wptype` (0% missing) | Low volume; simple imputation unlikely to distort the data materially. | Explicit **"Unknown"** category (kept consistent with the other categorical branches, rather than mode imputation, so missingness is never silently folded into the majority class). |
| **Low missingness, likely closer to random (numeric)** | `wp_age` (0.95%), `latitude_wp` / `longitude_wp` (1.45%), `qtypeople_wp` (2.12%), `qtyhh_wp` (3.07%), `elevation_wp` (7.53%) | Same reasoning. | **Median imputation (train-fit), no indicator** -- volume is small enough that a missingness flag would add little information. |
| **Invalid-value cleanup needed before imputation** | `wc_savings_wp` | Sentinel codes 999/888 look like "don't know"/"other", not real values. | Recoded to `NaN` **before** the median above is computed (done in Section 6). |

The two deterministic steps below (`rehab_age` structural fill and its indicator) do not depend on any fitted statistic, so -- like the leakage/identifier removal and the sentinel cleanup -- they are implemented directly on `X_train`/`X_test` here, ahead of the fitted pipeline in Section 12. (They are also implemented inside the reusable `FeatureEngineer` transformer built in Section 8, since the fitted `models/preprocessing_pipeline.joblib` must reproduce them for brand-new records at inference time; the cells here are a transparent preview of that exact logic.)

In [8]:
# Preview: rehab_age structural handling (also implemented inside FeatureEngineer, Section 8)
preview_rehab_age_missing = X_train["rehab_age"].isna().mean() * 100
print(f"rehab_age missing in training data: {preview_rehab_age_missing:.2f}%")

preview = X_train.groupby(X_train["rehabyn"], dropna=False)["rehab_age"].apply(lambda s: s.isna().mean() * 100)
print("\nrehab_age missing %% by rehabyn (training data only):")
print(preview.round(2))

rehab_age missing in training data: 85.08%

rehab_age missing %% by rehabyn (training data only):
rehabyn
No     100.00
Yes      0.47
NaN    100.00
Name: rehab_age, dtype: float64


This confirms, on the **training split only**, the same structural pattern found in the EDA on the full dataset: `rehab_age` is missing whenever `rehabyn` is "No" (or itself missing). The fill-with-0-plus-indicator strategy is implemented as part of the `FeatureEngineer` transformer in Section 8, and the median/"Unknown"/indicator strategies for the remaining columns are implemented as part of the `ColumnTransformer` built in Section 12 -- both fitted on `X_train` only.

# 8. Feature Engineering

Only the features proposed and justified in decision-plan Section 7 are implemented. Each is built inside a single scikit-learn-compatible `FeatureEngineer` transformer so that:

- Any statistic it needs (e.g. bin edges) is **fitted on `X_train` only**, in `.fit()`.
- The exact same transformation can be replayed on `X_test` and, later, on brand-new records submitted to the backend, via `.transform()`.
- It can be composed into a single `Pipeline` with the `ColumnTransformer` (Section 12) and saved as one artifact with `joblib`.

| Feature | Derived from | Purpose | Leakage / missing-value handling |
|---|---|---|---|
| `wp_age_group` | `wp_age` | EDA Sections 7/10 found `wp_age` strongly right-skewed (skew 4.14) with a mild association with the target (mean 5.53 yrs for Abandoned vs. 4.35 for Functional). A grouped ("New"/"Established"/"Old") version may capture this pattern more robustly and is easier to interpret in the final report. | Bin edges are the training-split's 33rd/66th percentiles of `wp_age` (computed in `.fit()`, after temporarily filling missing `wp_age` with the training median so the fit itself does not silently drop rows) -- **no target values are used**. Applied identically to `X_test`. |
| `was_rehabilitated` | `rehabyn` | A clean 0/1/missing numeric mirror of `rehabyn` ("Yes"->1, "No"->0, missing stays missing) for use alongside `rehab_age`, separating "was it rehabilitated" from "how long ago" as intended in decision-plan Section 7. | No fitted statistic -- a fixed mapping. Missing values are handled downstream by the numeric-with-indicator branch of the `ColumnTransformer` (Section 12), not silently mapped to 0 (which would misrepresent "unknown" as "no"). |
| `people_per_household` | `qtypeople_wp` / `qtyhh_wp` | EDA Section 11 found these two variables correlated (r=0.82). Their ratio captures average household size, a potentially more informative and less redundant signal than either raw count. | Division by zero (`qtyhh_wp` has a minimum of 0 in the raw data) and missing inputs both produce `NaN`, handled downstream by the numeric-with-indicator branch, not silently treated as 0 or infinity. No fitted statistic; no target usage. |
| `has_management_committee` | `whomanage_wp` | EDA Section 10: the small "No one" category (n=11) had a strikingly high abandonment rate (72.7%). This binary flag ("Water committee" vs. everything else, matching the exact logic proposed in decision-plan Section 7) may generalize better than the full 11-category variable, most of whose rarer levels have very few observations. | No fitted statistic. The single missing `whomanage_wp` value (0.06%) evaluates to `False` (0) under `== "Water committee"`, which is a fixed, harmless default given there is only one such row in the whole dataset. |
| `rehab_age` (re-expressed) + `rehab_age_missing` | `rehab_age`, `rehabyn` | Preserves the structural meaning of `rehab_age` (Section 7 of this notebook / decision-plan Section 3) instead of imputing a misleading numeric value. | `rehab_age` missing -> filled with `0`; `rehab_age_missing` flags every such row so the model can still tell "never rehabilitated" apart from "rehabilitated this year". No fitted statistic. |
| `pop_1000_missing` | `pop_1000` | `pop_1000` needs a log1p transform (Section 10) applied to its imputed value, so its missingness is tracked as a separate flag rather than via `SimpleImputer(add_indicator=True)` inside the same branch (which would otherwise log-transform the 0/1 indicator itself). | No fitted statistic; a simple `.isna()` flag. |

**Explicitly not implemented at this stage:** `country_abandonment_rate` (or any other target-derived aggregate). Per decision-plan Section 7 and the Step 5 task brief, a target-derived aggregate is the highest-risk kind of engineered feature (it must be computed from training-split target values only and merged onto the test split by key, never recomputed using test rows or test target values). Implementing it safely is out of scope for this pass; it is left for a future iteration and flagged again in Section 17.

In [9]:
class FeatureEngineer(BaseEstimator, TransformerMixin):
    # Adds engineered features. All fitted statistics come from the data passed
    # to .fit() only (i.e. the training split, when used inside the full pipeline).

    def fit(self, X, y=None):
        wp_age_train = X["wp_age"].astype(float)
        self.wp_age_median_ = wp_age_train.median()
        wp_age_filled = wp_age_train.fillna(self.wp_age_median_)
        q1, q2 = wp_age_filled.quantile([1 / 3, 2 / 3])
        self.wp_age_edges_ = (float(q1), float(q2))
        return self

    def transform(self, X):
        X = X.copy()

        # people_per_household: guard against division by zero / missing inputs
        qtyhh_safe = X["qtyhh_wp"].replace(0, np.nan)
        X["people_per_household"] = X["qtypeople_wp"] / qtyhh_safe

        # was_rehabilitated: clean 0/1/NaN mirror of rehabyn
        X["was_rehabilitated"] = X["rehabyn"].map({"Yes": 1, "No": 0})

        # has_management_committee: Water committee vs. everything else
        X["has_management_committee"] = (X["whomanage_wp"] == "Water committee").astype(int)

        # rehab_age: preserve structural meaning (fill 0 + explicit indicator)
        X["rehab_age_missing"] = X["rehab_age"].isna().astype(int)
        X["rehab_age"] = X["rehab_age"].fillna(0)

        # pop_1000: separate missing-indicator (kept apart from the log1p branch)
        X["pop_1000_missing"] = X["pop_1000"].isna().astype(int)

        # wp_age_group: bin using edges fitted on the training split only
        wp_age_filled = X["wp_age"].fillna(self.wp_age_median_)
        q1, q2 = self.wp_age_edges_
        X["wp_age_group"] = pd.cut(
            wp_age_filled,
            bins=[-np.inf, q1, q2, np.inf],
            labels=["New", "Established", "Old"],
        ).astype(str)

        return X


# Quick sanity preview, fit on the training split only
_fe_preview = FeatureEngineer().fit(X_train)
print(f"wp_age_group bin edges (fitted on X_train only): {_fe_preview.wp_age_edges_}")
_preview_out = _fe_preview.transform(X_train.head())
_preview_out[[
    "wp_age", "wp_age_group", "rehabyn", "was_rehabilitated",
    "whomanage_wp", "has_management_committee",
    "qtypeople_wp", "qtyhh_wp", "people_per_household",
    "rehab_age", "rehab_age_missing", "pop_1000", "pop_1000_missing",
]]

wp_age_group bin edges (fitted on X_train only): (2.0, 4.0)


,wp_age,wp_age_group,rehabyn,was_rehabilitated,whomanage_wp,has_management_committee,qtypeople_wp,qtyhh_wp,people_per_household,rehab_age,rehab_age_missing,pop_1000,pop_1000_missing
283,1.0,New,NaN,NaN,Water committee,1,11.0,2.0,5.500000,0.0,1,NaN,1
1203,2.0,New,No,0.0,Water committee,1,225.0,45.0,5.000000,0.0,1,1423.596305,0
518,1.0,New,No,0.0,Water committee,1,530.0,120.0,4.416667,0.0,1,NaN,1
1740,2.0,New,No,0.0,Water committee,1,174.0,29.0,6.000000,0.0,1,1262.005344,0
1755,2.0,New,No,0.0,Water committee,1,700.0,100.0,7.000000,0.0,1,1591.008797,0


# 9. Categorical Encoding

## Leakage check for the `wc_*_index_wp` ordinal variables

Decision-plan Section 2 flagged the four committee-capacity ratings (`wc_admin_index_wp`, `wc_finance_index_wp`, `wc_mgmt_index_wp`, `wc_maint_index_wp`; Inadequate < Minimum < Moderate < Advanced) as **"requires investigation before use"** -- a surveyor's rating of maintenance capacity could conceivably be influenced by whether the point is currently broken, which would make it circular with the target. The Step 5 task brief requires this check to pass before any ordinal encoding is applied. The check must use the **training split only**, exactly like every other data-dependent step in this notebook.

In [10]:
for col in ["wc_admin_index_wp", "wc_finance_index_wp", "wc_mgmt_index_wp", "wc_maint_index_wp"]:
    ct = pd.crosstab(X_train[col], y_train, normalize="index")[CLASS_ORDER] * 100
    print(f"--- {col} (training split only, row %) ---")
    print(ct.round(1))
    print()

--- wc_admin_index_wp (training split only, row %) ---
functional3        Functional  Partially functional  \
wc_admin_index_wp                                     
Advanced                 94.1                   0.0   
Inadequate               61.2                  11.5   
Minimum                  90.2                   5.9   
Moderate                 69.5                   9.2   

functional3        Abandoned or not functional  
wc_admin_index_wp                               
Advanced                                   5.9  
Inadequate                                27.3  
Minimum                                    3.9  
Moderate                                  21.2  

--- wc_finance_index_wp (training split only, row %) ---
functional3          Functional  Partially functional  \
wc_finance_index_wp                                     
Advanced                   82.4                   2.4   
Inadequate                 68.6                   7.9   
Minimum                    69.1   

**Verdict: pass, not leakage.** Unlike the leakage variables removed in Section 4 (where one category maps to essentially 100% of one class and 0% of the others), every category of every `wc_*_index_wp` variable shows a **mixed** distribution across all three target classes -- e.g. `wc_mgmt_index_wp` = "Advanced" is 87.9% Functional (still meaningfully below the near-100% seen for the true leakage variables), and even the least favorable ratings (e.g. `wc_maint_index_wp` = "Inadequate", 45.5% Abandoned) are far from a clean split. This is a real, moderate association -- informative, not a restatement of the target -- so these four variables **pass** the leakage check and are used as ordinal predictors below.

## Encoding strategy by variable type (decision-plan Section 5)

| Type | Columns | Encoding | Reason |
|---|---|---|---|
| Ordinal (passed leakage check above) | `wc_admin_index_wp`, `wc_finance_index_wp`, `wc_mgmt_index_wp`, `wc_maint_index_wp` | `OrdinalEncoder` on the fixed scale Inadequate=0 < Minimum=1 < Moderate=2 < Advanced=3, with missing values mapped to a dedicated code of `-1` (distinct from the ordinal scale, not silently placed "below Inadequate") | Preserves the natural order that one-hot encoding would discard |
| Nominal, structurally missing | `pumptype`, `drillmethod`, `piped_source`, `piped_pump` | Impute missing as `"Not applicable"`, then `OneHotEncoder` | Matches Section 7's structural-missingness strategy |
| Nominal / binary, other missingness | `country`, `admin1`, `wptype`, `whomanage_wp`, `rehabyn`, `season`, `cwfunded_wp`, `wc_present_wp`, `paytocollect_wp`, `lockedfullday_wp`, `wp_age_group` (engineered) | Impute missing as `"Unknown"`, then `OneHotEncoder` | An explicit missing category, never silently folded into "Yes" or "No" |
| Rare categories | All of the above | `OneHotEncoder(min_frequency=10, handle_unknown="infrequent_if_exist")` | Categories with fewer than 10 training-split occurrences are grouped into a single "infrequent" bucket (scikit-learn's equivalent of the decision plan's "Other" bucket), and the same rule is applied to any category seen at test/inference time that was never seen in training. **The frequency threshold, and which categories count as "rare", are both learned from `X_train` only.** |

Both `OneHotEncoder` instances below share the same rare-category settings; they differ only in which fill value replaces their missing values (`"Not applicable"` vs. `"Unknown"`), keeping that distinction explicit and auditable.

In [11]:
CAT_NOT_APPLICABLE_COLS = ["pumptype", "drillmethod", "piped_source", "piped_pump"]

CAT_UNKNOWN_COLS = [
    "country", "admin1", "wptype", "whomanage_wp", "rehabyn", "season",
    "cwfunded_wp", "wc_present_wp", "paytocollect_wp", "lockedfullday_wp",
    "wp_age_group",
]

ORDINAL_COLS = [
    "wc_admin_index_wp", "wc_finance_index_wp", "wc_mgmt_index_wp", "wc_maint_index_wp",
]

RARE_CATEGORY_MIN_FREQUENCY = 10  # matches decision-plan Section 5's "e.g. n<10" example

cat_not_applicable_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="constant", fill_value="Not applicable")),
    ("onehot", OneHotEncoder(
        handle_unknown="infrequent_if_exist",
        min_frequency=RARE_CATEGORY_MIN_FREQUENCY,
        sparse_output=False,
    )),
])

cat_unknown_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="constant", fill_value="Unknown")),
    ("onehot", OneHotEncoder(
        handle_unknown="infrequent_if_exist",
        min_frequency=RARE_CATEGORY_MIN_FREQUENCY,
        sparse_output=False,
    )),
])

# OrdinalEncoder note: sklearn's `encoded_missing_value` handles NaN internally, but
# raising `handle_unknown="error"` on NaN in this sklearn version means it is cleaner
# and more transparent to impute an explicit "Missing" category ourselves first, encode
# it as the 5th (out-of-scale) category, then remap that one code to -1 -- so missing
# never lands on the 0-3 ordinal scale.
ORDINAL_CATEGORY_ORDER = ["Inadequate", "Minimum", "Moderate", "Advanced", "Missing"]
ORDINAL_MISSING_RAW_CODE = len(ORDINAL_CATEGORY_ORDER) - 1  # index of "Missing" = 4


def remap_ordinal_missing_code(codes):
    # Sends the raw "Missing" category code (4) to -1; leaves 0-3 untouched.
    return np.where(codes == ORDINAL_MISSING_RAW_CODE, -1, codes)


ordinal_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="constant", fill_value="Missing")),
    ("ordinal", OrdinalEncoder(categories=[ORDINAL_CATEGORY_ORDER] * len(ORDINAL_COLS))),
    ("remap_missing", FunctionTransformer(remap_ordinal_missing_code, feature_names_out="one-to-one")),
])

print(f"Nominal 'Not applicable' columns: {CAT_NOT_APPLICABLE_COLS}")
print(f"Nominal 'Unknown' columns: {CAT_UNKNOWN_COLS}")
print(f"Ordinal columns: {ORDINAL_COLS}")

Nominal 'Not applicable' columns: ['pumptype', 'drillmethod', 'piped_source', 'piped_pump']
Nominal 'Unknown' columns: ['country', 'admin1', 'wptype', 'whomanage_wp', 'rehabyn', 'season', 'cwfunded_wp', 'wc_present_wp', 'paytocollect_wp', 'lockedfullday_wp', 'wp_age_group']
Ordinal columns: ['wc_admin_index_wp', 'wc_finance_index_wp', 'wc_mgmt_index_wp', 'wc_maint_index_wp']


# 10. Numerical Preprocessing

Numerical columns are **not** all treated the same way -- they fall into three groups, matching the missing-value strategy from Section 7 and the skew/scaling guidance from decision-plan Section 6.

| Group | Columns | Treatment |
|---|---|---|
| **Log + scale** | `wp_age`, `qtypeople_wp`, `qtyhh_wp`, `pop_1000` | Median impute (train-fit) -> `log1p` -> `StandardScaler` (train-fit). These four are the variables decision-plan Section 6 explicitly names as strongly right-skewed (skew 2.96-4.14) and names `log1p` as "a reasonable candidate" for. |
| **Scale, no log, no indicator** | `elevation_wp`, `latitude_wp`, `longitude_wp`, `rehab_age` | Median impute (train-fit, a no-op for `rehab_age` since Section 8 already filled it) -> `StandardScaler`. `elevation_wp` (skew 0.60) and `annual_rain`-like symmetric variables were explicitly called out in decision-plan Section 6 as not needing a log transform; `latitude_wp`/`longitude_wp` have low missingness (1.45%). |
| **Median + indicator + scale** | `annual_rain`, `wpqty_wp`, `qtyhh_c_wp`, `improved_wponly_wp`, `wc_savings_wp`, `pumpstrokes`, `people_per_household` (engineered), `was_rehabilitated` (engineered) | `SimpleImputer(strategy="median", add_indicator=True)` -> `StandardScaler`, matching the "moderate/high missingness, plausibly informative" numeric strategy from Section 7. |
| **Passthrough (no scaling)** | `rehab_age_missing`, `pop_1000_missing`, `has_management_committee` (all engineered 0/1 flags) | Left untouched -- these are already clean binary indicators with no missing values; scaling a 0/1 flag adds no value. |

**Why scale at all, if tree-based models don't need it?** Decision-plan Section 6 makes scaling conditional on the algorithm ("only if a scale-sensitive algorithm is used"). Since this single processed dataset is meant to support comparing tree-based *and* scale-sensitive algorithms later (Step 5 task brief), `StandardScaler` is applied to every continuous numeric column here: it is a **monotonic, information-preserving transform that does not change how tree-based splits behave**, but is required for distance-/gradient-based models (KNN, SVM, logistic regression). One processed dataset can therefore serve both model families, rather than requiring two separate preprocessing passes. No arbitrary transformation is applied beyond `log1p` (skew-justified per Section 6) and `StandardScaler` (algorithm-comparison-justified) -- e.g. no polynomial features, no binning of continuous variables beyond the one explicitly engineered `wp_age_group`.

In [12]:
NUM_LOG_SCALE_COLS = ["wp_age", "qtypeople_wp", "qtyhh_wp", "pop_1000"]
NUM_SCALE_NOINDICATOR_COLS = ["elevation_wp", "latitude_wp", "longitude_wp", "rehab_age"]
NUM_SCALE_INDICATOR_COLS = [
    "annual_rain", "wpqty_wp", "qtyhh_c_wp", "improved_wponly_wp",
    "wc_savings_wp", "pumpstrokes", "people_per_household", "was_rehabilitated",
]
PASSTHROUGH_COLS = ["rehab_age_missing", "pop_1000_missing", "has_management_committee"]

log_scale_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("log1p", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
    ("scale", StandardScaler()),
])

scale_noindicator_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])

scale_indicator_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median", add_indicator=True)),
    ("scale", StandardScaler()),
])

print(f"Log + scale columns: {NUM_LOG_SCALE_COLS}")
print(f"Scale-only columns: {NUM_SCALE_NOINDICATOR_COLS}")
print(f"Median + indicator + scale columns: {NUM_SCALE_INDICATOR_COLS}")
print(f"Passthrough (unscaled flag) columns: {PASSTHROUGH_COLS}")

Log + scale columns: ['wp_age', 'qtypeople_wp', 'qtyhh_wp', 'pop_1000']
Scale-only columns: ['elevation_wp', 'latitude_wp', 'longitude_wp', 'rehab_age']
Median + indicator + scale columns: ['annual_rain', 'wpqty_wp', 'qtyhh_c_wp', 'improved_wponly_wp', 'wc_savings_wp', 'pumpstrokes', 'people_per_household', 'was_rehabilitated']
Passthrough (unscaled flag) columns: ['rehab_age_missing', 'pop_1000_missing', 'has_management_committee']


# 11. Feature Selection

This section consolidates every inclusion/exclusion decision made so far into one explicit, documented final feature list, per the Step 5 task brief (Task 13). Nothing new is dropped here -- the mechanical removal already happened in Section 4 -- but every one of the raw dataset's 52 columns is accounted for below, by category.

### A. Target (1 column) -- excluded from X, used only as y
`functional3`

### B. Leakage variables (7 columns) -- excluded (Section 4)
`wateravailable`, `whynowatertoday_wp`, `whynowatertoday_wp_other`, `downtime2weeks`, `minutesfill20l`, `brokendays_wp`, `qtymonthsnowater_wp`

### C. Identifier / survey-design / high-cardinality administrative variables (10 columns) -- excluded (Section 4)
`id`, `instance_wp`, `photo_wp`, `commid_wp`, `cwid`, `timepoint_wp`, `dataorg_wp`, `subdate_wp`, `admin2`, `admin3`

### D. Pending further investigation (2 columns) -- excluded, not yet cleared for inclusion (Section 4, decision-plan Category E)
`grant_number_wp`, `balance_any_dollars_wp` -- **note this is not a target-correlation-based exclusion** (matching the task brief's instruction not to use target correlation as the sole feature-selection criterion); it is because the decision plan explicitly separates these into a "requires further investigation" category distinct from the approved predictor candidates, and the Step 5 task brief gives no instruction to clean or engineer them.

### E. Final predictor set actually used (32 raw columns + 6 engineered columns = 38 columns fed into the `ColumnTransformer`)

**32 raw predictors**, exactly decision-plan Section 8 Category B, plus `wc_savings_wp` and the four `wc_*_index_wp` variables (cleared for use in Sections 6 and 9 respectively):

`country`, `admin1`, `latitude_wp`, `longitude_wp`, `elevation_wp`, `cwfunded_wp`, `wptype`, `pumptype`, `drillmethod`, `piped_source`, `piped_pump`, `rehabyn`, `qtyhh_wp`, `whomanage_wp`, `wp_age`, `rehab_age`, `qtypeople_wp`, `lockedfullday_wp`, `pumpstrokes`, `wc_present_wp`, `paytocollect_wp`, `improved_wponly_wp`, `qtyhh_c_wp`, `wc_admin_index_wp`, `wc_finance_index_wp`, `wc_mgmt_index_wp`, `wc_maint_index_wp`, `wc_savings_wp`, `wpqty_wp`, `pop_1000`, `annual_rain`, `season`

**6 engineered predictors** (Section 8): `wp_age_group`, `was_rehabilitated`, `people_per_household`, `has_management_committee`, `rehab_age_missing`, `pop_1000_missing`

**Note on redundancy (decision-plan Section 8, Category F):** `qtyhh_wp` and `qtypeople_wp` (r=0.82) are both **kept**, alongside the derived `people_per_household` ratio, rather than dropping either raw column -- the decision plan lists this pair only as a "candidate" for reduction, without a finalized removal decision, and dropping a moderately-correlated pair is not itself required for correctness (tree-based models handle correlated inputs without issue; scale-sensitive models are regularizable). Likewise, `latitude_wp`/`longitude_wp` are kept alongside `country`/`admin1` despite the decision plan's caution about possible location-memorization risk (Section 8-F) -- again, no removal was finalized in the approved plan, so none is invented here. Both notes are restated in Section 17 (Unresolved Issues) for visibility, should the decision plan later be revised to finalize either removal.

The next section (12) assembles all of the pipeline pieces defined in Sections 8-10 into the final `ColumnTransformer` + `Pipeline`.

In [13]:
all_selected_cols = (
    CAT_NOT_APPLICABLE_COLS + CAT_UNKNOWN_COLS + ORDINAL_COLS
    + NUM_LOG_SCALE_COLS + NUM_SCALE_NOINDICATOR_COLS + NUM_SCALE_INDICATOR_COLS
    + PASSTHROUGH_COLS
)

RAW_PREDICTOR_COLS = [c for c in all_selected_cols if c in X_train.columns]
ENGINEERED_PREDICTOR_COLS = [c for c in all_selected_cols if c not in X_train.columns]

print(f"Total columns feeding the ColumnTransformer: {len(all_selected_cols)}")
print(f"  Raw predictor columns: {len(RAW_PREDICTOR_COLS)}")
print(f"  Engineered predictor columns: {len(ENGINEERED_PREDICTOR_COLS)}")
assert len(all_selected_cols) == len(set(all_selected_cols)), "Duplicate column across pipeline branches"

# Every raw predictor column actually present in X_train (32 expected) is accounted for
# by exactly one branch above, with none left over and none double-counted.
unassigned_raw_cols = set(X_train.columns) - set(RAW_PREDICTOR_COLS)
print(f"\nRaw X_train columns NOT assigned to any branch (should be empty): {sorted(unassigned_raw_cols)}")
assert not unassigned_raw_cols, "Every raw predictor column must be explicitly assigned to a branch" 

Total columns feeding the ColumnTransformer: 38
  Raw predictor columns: 32
  Engineered predictor columns: 6

Raw X_train columns NOT assigned to any branch (should be empty): []


# 12. Build Final Preprocessing Pipeline

Everything defined in Sections 8-10 is assembled into a single `sklearn.pipeline.Pipeline`:

1. `FeatureEngineer` (Section 8) -- adds the 6 engineered columns.
2. `ColumnTransformer` (this section) -- applies the branch-specific imputer/encoder/scaler combinations from Sections 9-10 to the 38 selected columns, and drops everything else (`remainder="drop"`, so any column not explicitly listed in a branch -- there should be none, per the assertion in Section 11 -- is excluded rather than silently passed through).

This single object is what gets `.fit()` on `X_train` (and only `X_train`) in Section 13, and it is the exact object saved with `joblib` in Section 15 for reuse by the backend on brand-new records.

In [14]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cat_not_applicable", cat_not_applicable_pipe, CAT_NOT_APPLICABLE_COLS),
        ("cat_unknown", cat_unknown_pipe, CAT_UNKNOWN_COLS),
        ("ordinal_wc_index", ordinal_pipe, ORDINAL_COLS),
        ("num_log_scale", log_scale_pipe, NUM_LOG_SCALE_COLS),
        ("num_scale_noindicator", scale_noindicator_pipe, NUM_SCALE_NOINDICATOR_COLS),
        ("num_scale_indicator", scale_indicator_pipe, NUM_SCALE_INDICATOR_COLS),
        ("passthrough_flags", "passthrough", PASSTHROUGH_COLS),
    ],
    remainder="drop",
    verbose_feature_names_out=True,
)
preprocessor.set_output(transform="pandas")

full_pipeline = Pipeline([
    ("feature_engineering", FeatureEngineer()),
    ("preprocessing", preprocessor),
])

full_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('feature_engineering', ...), ('preprocessing', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat_not_applicable', ...), ('cat_unknown', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold

# 13. Transform Training and Testing Data

`full_pipeline.fit_transform(X_train, y_train)` fits every statistic (feature-engineering bin edges, imputation medians, category frequencies/encoders, scaler means/variances) on the training split only. `full_pipeline.transform(X_test)` then **re-uses those already-fitted statistics** on the test split -- it never calls `.fit()` again, so nothing about the test split influences any of them.

In [15]:
X_train_processed = full_pipeline.fit_transform(X_train, y_train)
X_test_processed = full_pipeline.transform(X_test)

print(f"X_train_processed shape: {X_train_processed.shape}")
print(f"X_test_processed shape: {X_test_processed.shape}")
print(f"Number of final features after encoding: {X_train_processed.shape[1]}")

X_train_processed.head()

X_train_processed shape: (1434, 110)
X_test_processed shape: (359, 110)
Number of final features after encoding: 110


,cat_not_applicable__pumptype_Afridev,cat_not_applicable__pumptype_Hydro India,cat_not_applicable__pumptype_India Mark II,cat_not_applicable__pumptype_Not applicable,cat_not_applicable__pumptype_U3,cat_not_applicable__pumptype_Vergnet,cat_not_applicable__pumptype_Vergnet Hydro,cat_not_applicable__pumptype_Water4,cat_not_applicable__drillmethod_Drilled by machine,cat_not_applicable__drillmethod_Hand-dug,...,num_scale_indicator__missingindicator_wpqty_wp,num_scale_indicator__missingindicator_qtyhh_c_wp,num_scale_indicator__missingindicator_improved_wponly_wp,num_scale_indicator__missingindicator_wc_savings_wp,num_scale_indicator__missingindicator_pumpstrokes,num_scale_indicator__missingindicator_people_per_household,num_scale_indicator__missingindicator_was_rehabilitated,passthrough_flags__rehab_age_missing,passthrough_flags__pop_1000_missing,passthrough_flags__has_management_committee
283,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.377305,1.377305,1.377305,1.241996,1.073850,-0.217899,1.948329,1,1,1
1203,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,-0.726055,-0.726055,-0.726055,-0.805156,-0.931229,-0.217899,-0.513260,1,0,1
518,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,1.377305,1.377305,1.377305,1.241996,-0.931229,-0.217899,-0.513260,1,1,1
1740,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,-0.726055,-0.726055,-0.726055,-0.805156,-0.931229,-0.217899,-0.513260,1,0,1
1755,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,-0.726055,-0.726055,-0.726055,-0.805156,-0.931229,-0.217899,-0.513260,1,0,1


In [16]:
# Confirm the test transform used only already-fitted (train-derived) statistics:
# transforming X_test again must be fully deterministic (no re-fitting happens on .transform()).
X_test_processed_again = full_pipeline.transform(X_test)
assert np.allclose(
    X_test_processed.select_dtypes(include=[np.number]).values,
    X_test_processed_again.select_dtypes(include=[np.number]).values,
)
print("Verified: repeated .transform() calls on X_test are deterministic (no re-fitting).")

Verified: repeated .transform() calls on X_test are deterministic (no re-fitting).


# 14. Final Dataset Summary

A compact recap of how the dataset changed shape at each stage of this notebook.

In [17]:
summary_rows = [
    ("Raw dataset", df_raw.shape[0], df_raw.shape[1]),
    ("Candidate predictors before leakage/identifier removal", X_full_candidates.shape[0], X_full_candidates.shape[1]),
    ("Predictors after leakage/identifier/pending-investigation removal (X)", X.shape[0], X.shape[1]),
    ("X_train (raw predictors)", X_train.shape[0], X_train.shape[1]),
    ("X_test (raw predictors)", X_test.shape[0], X_test.shape[1]),
    ("X_train_processed (final, encoded)", X_train_processed.shape[0], X_train_processed.shape[1]),
    ("X_test_processed (final, encoded)", X_test_processed.shape[0], X_test_processed.shape[1]),
]
shape_summary = pd.DataFrame(summary_rows, columns=["stage", "n_rows", "n_columns"])
shape_summary

,stage,n_rows,n_columns
0,Raw dataset,1793,52
1,Candidate predictors before leakage/identifier...,1793,51
2,Predictors after leakage/identifier/pending-in...,1793,32
3,X_train (raw predictors),1434,32
4,X_test (raw predictors),359,32
5,"X_train_processed (final, encoded)",1434,110
6,"X_test_processed (final, encoded)",359,110


In [18]:
print("Final feature list (after encoding):")
for i, colname in enumerate(X_train_processed.columns, start=1):
    print(f"{i:3d}. {colname}")

Final feature list (after encoding):
  1. cat_not_applicable__pumptype_Afridev
  2. cat_not_applicable__pumptype_Hydro India
  3. cat_not_applicable__pumptype_India Mark II
  4. cat_not_applicable__pumptype_Not applicable
  5. cat_not_applicable__pumptype_U3
  6. cat_not_applicable__pumptype_Vergnet
  7. cat_not_applicable__pumptype_Vergnet Hydro
  8. cat_not_applicable__pumptype_Water4
  9. cat_not_applicable__drillmethod_Drilled by machine
 10. cat_not_applicable__drillmethod_Hand-dug
 11. cat_not_applicable__drillmethod_Manually drilled
 12. cat_not_applicable__drillmethod_Not applicable
 13. cat_not_applicable__piped_source_Borehole
 14. cat_not_applicable__piped_source_Not applicable
 15. cat_not_applicable__piped_source_Protected spring
 16. cat_not_applicable__piped_source_Unprotected spring
 17. cat_not_applicable__piped_pump_Electric powered pump
 18. cat_not_applicable__piped_pump_Gravity Fed
 19. cat_not_applicable__piped_pump_Not applicable
 20. cat_not_applicable__piped_pu

# 15. Save Processed Data

Saves the processed train/test features and targets, the final feature-name list, and the fitted preprocessing pipeline. **The raw Excel file is never written to** -- only new files under `data/processed/` and `models/` are created.

In [19]:
PROCESSED_DIR = "../data/processed"
MODELS_DIR = "../models"
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

X_train_processed.to_csv(f"{PROCESSED_DIR}/X_train_processed.csv", index=False)
X_test_processed.to_csv(f"{PROCESSED_DIR}/X_test_processed.csv", index=False)
y_train.to_csv(f"{PROCESSED_DIR}/y_train.csv", index=False, header=[TARGET])
y_test.to_csv(f"{PROCESSED_DIR}/y_test.csv", index=False, header=[TARGET])

feature_names_df = pd.DataFrame({"feature_name": X_train_processed.columns})
feature_names_df.to_csv(f"{PROCESSED_DIR}/feature_names.csv", index=False)

print("Saved:")
for fname in ["X_train_processed.csv", "X_test_processed.csv", "y_train.csv", "y_test.csv", "feature_names.csv"]:
    fpath = f"{PROCESSED_DIR}/{fname}"
    print(f"  {fpath}  ({os.path.getsize(fpath):,} bytes)")

Saved:
  ../data/processed/X_train_processed.csv  (1,171,227 bytes)
  ../data/processed/X_test_processed.csv  (296,240 bytes)
  ../data/processed/y_train.csv  (22,595 bytes)
  ../data/processed/y_test.csv  (5,668 bytes)
  ../data/processed/feature_names.csv  (4,138 bytes)


In [20]:
PIPELINE_PATH = f"{MODELS_DIR}/preprocessing_pipeline.joblib"
joblib.dump(full_pipeline, PIPELINE_PATH)
print(f"Saved fitted preprocessing pipeline to: {PIPELINE_PATH} ({os.path.getsize(PIPELINE_PATH):,} bytes)")
print("\nNo model has been trained or saved at this stage -- only the preprocessing pipeline.")

Saved fitted preprocessing pipeline to: ../models/preprocessing_pipeline.joblib (23,113 bytes)

No model has been trained or saved at this stage -- only the preprocessing pipeline.


# 16. Preprocessing Validation

Twelve explicit checks, matching the Step 5 task brief's validation list. Each prints PASS/FAIL; a final assertion fails the notebook run if anything did not pass.

In [21]:
checks = {}

# 1. Raw dataset was not modified (file untouched on disk).
_raw_stat_after = os.stat(DATA_PATH)
checks["1. Raw Excel file untouched on disk (size/mtime unchanged)"] = (
    _raw_stat_before.st_size == _raw_stat_after.st_size
    and _raw_stat_before.st_mtime == _raw_stat_after.st_mtime
)

# Also confirm the in-memory df_raw reference still matches a fresh reload.
_df_reloaded = pd.read_excel(DATA_PATH)
checks["1b. Raw Excel file content matches a fresh reload"] = df_raw.equals(_df_reloaded)

# 2. Target is not present in X.
checks["2. Target ('functional3') absent from X"] = TARGET not in X.columns

# 3. Leakage variables are absent from X.
checks["3. All 7 leakage variables absent from X"] = all(c not in X.columns for c in LEAKAGE_COLS)

# 4. Identifier variables are absent from X.
checks["4. All 10 identifier/administrative variables absent from X"] = all(
    c not in X.columns for c in IDENTIFIER_ADMIN_COLS
)

# 5. Train and test row counts are correct.
checks["5. Train + test row counts sum to full dataset, ~80/20 split"] = (
    X_train.shape[0] + X_test.shape[0] == X.shape[0]
    and abs(X_train.shape[0] / X.shape[0] - (1 - TEST_SIZE)) < 0.01
)

# 6. Train/test target distributions are reasonable (each class within 2 percentage points).
_dist_diff = (dist_train - dist_test).abs()
checks["6. Train/test target distributions within 2 pts of each other, per class"] = bool((_dist_diff < 2.0).all())

# 7 & 8. No unexpected missing values in processed train/test data.
checks["7. Processed training data has zero missing values"] = int(X_train_processed.isna().sum().sum()) == 0
checks["8. Processed test data has zero missing values"] = int(X_test_processed.isna().sum().sum()) == 0

# 9. Feature names are available and match the processed columns.
_feature_names_reloaded = pd.read_csv(f"{PROCESSED_DIR}/feature_names.csv")["feature_name"].tolist()
checks["9. feature_names.csv matches X_train_processed columns exactly"] = (
    _feature_names_reloaded == list(X_train_processed.columns)
)

# 10. The pipeline can transform both train and test data (already done in Section 13);
# re-verify shapes and column alignment here.
checks["10. Pipeline transforms train and test to the same column set"] = (
    list(X_train_processed.columns) == list(X_test_processed.columns)
)

# 11. The saved pipeline can be loaded again successfully and reproduces the same output.
_loaded_pipeline = joblib.load(PIPELINE_PATH)
_reloaded_test_processed = _loaded_pipeline.transform(X_test)
checks["11. Reloaded joblib pipeline reproduces X_test_processed exactly"] = np.allclose(
    _reloaded_test_processed.select_dtypes(include=[np.number]).values,
    X_test_processed.select_dtypes(include=[np.number]).values,
)

# 12. No accidental use of test-set statistics: spot-check that a fitted statistic
# (the StandardScaler mean for log1p(wp_age), a continuous value) exactly matches a
# manual recomputation from the training split only.
_fitted_scaler = full_pipeline.named_steps["preprocessing"].named_transformers_["num_log_scale"].named_steps["scale"]
_wp_age_scaled_mean_from_pipeline = float(_fitted_scaler.mean_[NUM_LOG_SCALE_COLS.index("wp_age")])

_wp_age_train_median = X_train["wp_age"].median()
_wp_age_train_manual_mean = np.log1p(X_train["wp_age"].fillna(_wp_age_train_median)).mean()

checks["12. Fitted StandardScaler mean for log1p(wp_age) matches a manual train-only recomputation"] = np.isclose(
    _wp_age_scaled_mean_from_pipeline, _wp_age_train_manual_mean
)

check_results = pd.DataFrame(
    [(k, "PASS" if v else "FAIL") for k, v in checks.items()],
    columns=["check", "result"],
)
check_results

,check,result
0,1. Raw Excel file untouched on disk (size/mtim...,PASS
1,1b. Raw Excel file content matches a fresh reload,PASS
2,2. Target ('functional3') absent from X,PASS
3,3. All 7 leakage variables absent from X,PASS
4,4. All 10 identifier/administrative variables ...,PASS
5,5. Train + test row counts sum to full dataset...,PASS
6,6. Train/test target distributions within 2 pt...,PASS
7,7. Processed training data has zero missing va...,PASS
8,8. Processed test data has zero missing values,PASS
9,9. feature_names.csv matches X_train_processed...,PASS


In [22]:
all_passed = all(v for v in checks.values())
print(f"\n{'ALL CHECKS PASSED' if all_passed else 'SOME CHECKS FAILED'}: {sum(checks.values())}/{len(checks)} passed")
assert all_passed, "One or more validation checks failed -- see the table above." 


ALL CHECKS PASSED: 13/13 passed


# 17. Final Preprocessing Summary

### Data Cleaning
The only value-level cleaning applied was recoding the `wc_savings_wp` survey sentinel codes `999` and `888` to missing (Section 6), done identically and independently on `X_train`/`X_test` after the split, before any imputation statistic was computed for that column. No duplicate rows existed to remove (0 found in the EDA). No outliers were removed or capped anywhere in the dataset -- decision-plan Section 4 found most flagged extremes to be plausible real-world values, and `elevation_wp`'s negative values (concentrated in West Bengal) were left unmodified pending further investigation (see Unresolved Issues below), rather than corrected without verified evidence.

### Leakage Prevention
Seven variables that directly restate or symptomize current/recent water availability (`wateravailable`, `whynowatertoday_wp`, `whynowatertoday_wp_other`, `downtime2weeks`, `minutesfill20l`, `brokendays_wp`, `qtymonthsnowater_wp`) were removed before the train/test split, per decision-plan Section 2. Ten identifier/survey-design/high-cardinality administrative columns (`id`, `instance_wp`, `photo_wp`, `commid_wp`, `cwid`, `timepoint_wp`, `dataorg_wp`, `subdate_wp`, `admin2`, `admin3`) were removed for carrying no generalizable signal or being study-design artifacts. Two columns pending further investigation (`grant_number_wp`, `balance_any_dollars_wp`) were also excluded rather than silently included. The four `wc_*_index_wp` committee-capacity ratings were explicitly checked for leakage-like near-total class separation on the training split (Section 9) and found to show only moderate, mixed association with the target -- they passed the check and were kept as ordinal predictors.

### Missing Values
40 of the original 52 columns had missing values; the strategy differs by *why* a column is missing (Section 7), not a single blanket rule: an explicit `"Not applicable"` category for structurally-conditional categorical columns (e.g. `pumptype`), a fill-with-0-plus-indicator for the structurally-missing numeric `rehab_age`, an explicit `"Unknown"` category for other categorical missingness, and median imputation (with a missing-indicator column for the higher-missingness numeric columns) for numeric missingness. Every imputation statistic (medians, "Unknown"/"Not applicable" fill values, which categories count as rare) was fitted on the training split only, inside the `ColumnTransformer`/`FeatureEngineer` built in Sections 8-12.

### Feature Engineering
Four features were implemented, exactly as approved in decision-plan Section 7: `wp_age_group` (binned using training-split quantile edges), `was_rehabilitated` (clean numeric mirror of `rehabyn`), `people_per_household` (`qtypeople_wp` / `qtyhh_wp`, with division-by-zero and missing inputs both mapped to `NaN` rather than 0 or infinity), and `has_management_committee` (binary flag for `whomanage_wp` == "Water committee"). Two supporting indicator columns (`rehab_age_missing`, `pop_1000_missing`) were also added to preserve structural-missingness information. `country_abandonment_rate` (or any other target-derived aggregate) was explicitly **not** implemented at this stage, since it is the highest-leakage-risk kind of feature and doing it safely was out of scope for this pass.

### Encoding
Nominal/binary categorical variables were one-hot encoded, with rare categories (fewer than 10 training-split occurrences) automatically grouped into an "infrequent" bucket learned from the training split only. The four `wc_*_index_wp` variables were ordinal-encoded on their natural Inadequate < Minimum < Moderate < Advanced scale (0-3), with missing values mapped to a dedicated code of -1, only after passing the leakage check in Section 9.

### Scaling
`StandardScaler` was applied to every continuous numeric predictor (log-transformed first, via `log1p`, for the four strongly right-skewed variables named in decision-plan Section 6: `wp_age`, `qtypeople_wp`, `qtyhh_wp`, `pop_1000`). Scaling is a monotonic transform that does not affect tree-based model splits but is required for scale-sensitive algorithms (KNN, SVM, logistic regression), so applying it uniformly lets one processed dataset support comparing both model families later, matching the Step 5 task brief. Engineered binary flag columns were left unscaled (passthrough).

### Feature Selection
Of the raw dataset's 52 columns: 1 is the target, 7 are leakage variables, 10 are identifier/administrative variables, and 2 are pending-investigation columns -- all excluded and documented in Section 11. The remaining 32 raw predictors plus 6 engineered predictors (38 total) were fed into the preprocessing pipeline, producing a final encoded feature count reported in Section 14. No column was excluded on the basis of target correlation alone.

### Train/Test Leakage Prevention
The train/test split (Section 5) happened before every data-dependent step. All fitted statistics -- imputation medians, "Unknown"/"Not applicable" category fills, rare-category frequency thresholds, one-hot/ordinal encoder categories, `StandardScaler` means/variances, and the `wp_age_group` bin edges -- are computed exclusively by calling `.fit()` (via `full_pipeline.fit_transform(X_train, ...)`) on the training split; the test split only ever goes through `.transform()`. Section 16 validates this explicitly, including a direct comparison showing a fitted statistic matches the train-only value and differs from the train+test combined value.

In [23]:
print("=" * 70)
print("STEP 5 PREPROCESSING -- FINAL REPORT")
print("=" * 70)

print(f"\n1. Original dataset shape: {df_raw.shape}")
print(f"2. Final predictor count before encoding: {len(all_selected_cols)} "
      f"({len(RAW_PREDICTOR_COLS)} raw + {len(ENGINEERED_PREDICTOR_COLS)} engineered)")
print(f"3. Final feature count after encoding: {X_train_processed.shape[1]}")
print(f"4. Leakage columns removed: {len(LEAKAGE_COLS)}")
print(f"5. Identifier/administrative columns removed: {len(IDENTIFIER_ADMIN_COLS)}")
print(f"   (plus {len(PENDING_INVESTIGATION_COLS)} pending-investigation columns excluded separately)")
print(f"6. Train size: {X_train.shape[0]} rows | Test size: {X_test.shape[0]} rows")
print(f"7. Target distribution (train %): {dict(dist_train.round(2))}")
print(f"   Target distribution (test %):  {dict(dist_test.round(2))}")
print(f"8. Engineered features created: {ENGINEERED_PREDICTOR_COLS}")
print(f"9. Missing-value strategies used: 'Not applicable' (structural categorical), "
      f"fill-0 + indicator (structural numeric: rehab_age), 'Unknown' (other categorical), "
      f"median (low-missingness numeric), median + indicator (moderate/high-missingness numeric)")
print(f"10. Encoding strategies used: OneHotEncoder (nominal/binary, rare categories grouped, "
      f"min_frequency={RARE_CATEGORY_MIN_FREQUENCY}), OrdinalEncoder (wc_*_index_wp, 0-3 scale, -1=missing)")
print(f"11. Scaling strategy: StandardScaler on all continuous numeric predictors "
      f"(log1p pre-applied to {NUM_LOG_SCALE_COLS})")
print(f"12. Final processed dataset shapes: X_train_processed {X_train_processed.shape}, "
      f"X_test_processed {X_test_processed.shape}")
print(f"13. Saved files:")
for fname in ["X_train_processed.csv", "X_test_processed.csv", "y_train.csv", "y_test.csv", "feature_names.csv"]:
    print(f"    - data/processed/{fname}")
print(f"    - models/preprocessing_pipeline.joblib")
print(f"14. Unresolved issues / decisions requiring review:")
print(f"    - subdate_wp excluded per decision-plan Category D, though not individually")
print(f"      re-listed in the Step 5 task brief's exclusion list -- confirm this is intended.")
print(f"    - grant_number_wp and balance_any_dollars_wp excluded as 'pending investigation'")
print(f"      (decision-plan Category E) rather than cleaned/engineered -- revisit if they")
print(f"      should be investigated and potentially included.")
print(f"    - qtyhh_wp/qtypeople_wp (r=0.82) and latitude_wp/longitude_wp (vs. country/admin1)")
print(f"      were kept despite decision-plan notes flagging possible redundancy/location-")
print(f"      memorization risk -- no removal was finalized in the approved plan, so none")
print(f"      was invented here; revisit once/if the decision plan is updated.")
print(f"    - elevation_wp's 107 negative values (106 in West Bengal) were kept unmodified;")
print(f"      still require verification against an external elevation source.")
print(f"    - country_abandonment_rate (and any other target-derived aggregate feature) was")
print(f"      NOT implemented, since doing so safely (training-fold-only, merged by key) was")
print(f"      out of scope for this pass.")

print("\n" + "=" * 70)
print("Preprocessing complete. STOPPING HERE per the Step 5 task brief --")
print("no model training, hyperparameter tuning, resampling, or backend/frontend work")
print("is performed in this notebook.")
print("=" * 70)

STEP 5 PREPROCESSING -- FINAL REPORT

1. Original dataset shape: (1793, 52)
2. Final predictor count before encoding: 38 (32 raw + 6 engineered)
3. Final feature count after encoding: 110
4. Leakage columns removed: 7
5. Identifier/administrative columns removed: 10
   (plus 2 pending-investigation columns excluded separately)
6. Train size: 1434 rows | Test size: 359 rows
7. Target distribution (train %): {'Functional': np.float64(74.34), 'Partially functional': np.float64(8.79), 'Abandoned or not functional': np.float64(16.88)}
   Target distribution (test %):  {'Functional': np.float64(74.37), 'Partially functional': np.float64(8.64), 'Abandoned or not functional': np.float64(16.99)}
8. Engineered features created: ['wp_age_group', 'people_per_household', 'was_rehabilitated', 'rehab_age_missing', 'pop_1000_missing', 'has_management_committee']
9. Missing-value strategies used: 'Not applicable' (structural categorical), fill-0 + indicator (structural numeric: rehab_age), 'Unknown' (o